# MBTiles Conversion Test

Test converting a single 500m sub-cell area into an MBTiles file for SurveyCTO offline use.

**3-step workflow** (per the GDAL recipe):

1. **Download** high-res satellite tiles for the sub-cell bbox at max zoom (e.g. 19) → GeoTIFF
2. **Translate** GeoTIFF → MBTiles at base zoom
3. **Add overviews** (lower zoom levels) so the file is usable when zoomed out

Equivalent GDAL CLI commands (shown for reference):
```bash
gdal_translate -of MBTiles high_res.tif map.mbtiles
gdaladdo -r nearest map.mbtiles 2 4 8 16
gdalinfo map.mbtiles
```

Since GDAL CLI is not installed in this venv, we use **rasterio** (which bundles GDAL 3.12) to call the same drivers from Python.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling

from src.utils.config_loader import load_config, get_data_dir
from src.data_processing.load_boundaries import load_selected_subcells, validate_crs

print('rasterio:', rasterio.__version__, '| GDAL:', rasterio.__gdal_version__)

In [ ]:
config = load_config()
data_dir = get_data_dir(config)

# Test output dir (kept local — not pushed to Drive)
OUT_DIR = PROJECT_ROOT / 'tmp' / 'mbtiles_test'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Imagery source — same Google Hybrid tiles used in 08_surveycto_test
TILE_SOURCE = 'https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}'
MAX_ZOOM = 19  # max for Google satellite/hybrid

print('data_dir:', data_dir)
print('output:  ', OUT_DIR)

## Pick one sub-cell as the test target

We grab a single 500m sub-cell from `selected_subcells_500m.gpkg` and use its bounds (with a small pad).

In [ ]:
selected = load_selected_subcells(data_dir)
selected = validate_crs(selected)
print(f'Loaded {len(selected)} selected sub-cells (CRS: {selected.crs})')

# Take the first sampled (not replacement) sub-cell as the test
test_row = selected[selected['sample_status'] == 'sampled'].iloc[[0]].copy()
print('\nTest sub-cell:')
for col in ['5km_id', 'grid_id', 'ward_name', 'selection_role', 'building_count']:
    if col in test_row.columns:
        print(f'  {col}: {test_row.iloc[0][col]}')

# Reproject to Web Mercator (what the tile source uses)
test_3857 = test_row.to_crs(epsg=3857)
minx, miny, maxx, maxy = test_3857.total_bounds

# Pad bounds by 10% so the cell isn't right at the edge of the raster
pad_x = (maxx - minx) * 0.1
pad_y = (maxy - miny) * 0.1
west, south, east, north = minx - pad_x, miny - pad_y, maxx + pad_x, maxy + pad_y
print(f'\nBounds (EPSG:3857, padded): {west:.1f}, {south:.1f}, {east:.1f}, {north:.1f}')
print(f'Width:  {(east-west):.0f} m')
print(f'Height: {(north-south):.0f} m')

## Step 1 — Download tiles at max zoom

`contextily.bounds2raster` downloads all tiles covering the bbox at the requested zoom and stitches them into a georeferenced GeoTIFF.

In [ ]:
tif_path = OUT_DIR / 'high_res.tif'

img, ext = cx.bounds2raster(
    west, south, east, north,
    str(tif_path),
    zoom=MAX_ZOOM,
    source=TILE_SOURCE,
    ll=False,  # bounds are already in Web Mercator, not lat/lon
)

size_mb = tif_path.stat().st_size / 1e6
print(f'Saved: {tif_path}')
print(f'Size:  {size_mb:.2f} MB')
print(f'Shape: {img.shape}')

with rasterio.open(tif_path) as src:
    print(f'CRS:   {src.crs}')
    print(f'Bands: {src.count}, dtype: {src.dtypes[0]}')
    print(f'Size:  {src.width} x {src.height} px')

In [ ]:
# Sanity check — plot the downloaded raster with the sub-cell overlaid
fig, ax = plt.subplots(figsize=(8, 8))
with rasterio.open(tif_path) as src:
    from rasterio.plot import show
    show(src, ax=ax)
test_3857.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=2)
ax.set_title('Downloaded GeoTIFF + sub-cell overlay')
ax.set_axis_off()
plt.show()

## Step 2 — Convert GeoTIFF → MBTiles

CLI equivalent: `gdal_translate -of MBTiles high_res.tif map.mbtiles`

MBTiles requirements: input must be in **EPSG:3857** (Web Mercator) with **3 bands (RGB) or 4 bands (RGBA)**, dtype `uint8`. Tiles from `bounds2raster` already meet this.

In [ ]:
mbtiles_path = OUT_DIR / 'map.mbtiles'
if mbtiles_path.exists():
    mbtiles_path.unlink()  # MBTILES driver won't overwrite

with rasterio.open(tif_path) as src:
    profile = src.profile.copy()
    profile.update(
        driver='MBTILES',
        # MBTILES driver options
        TILE_FORMAT='PNG',
    )
    # MBTILES needs uint8 RGB(A) — drop alpha if present and not needed
    data = src.read()
    print(f'Source bands: {src.count}, writing {data.shape[0]} bands')
    with rasterio.open(mbtiles_path, 'w', **profile) as dst:
        dst.write(data)

print(f'Wrote: {mbtiles_path}')
print(f'Size:  {mbtiles_path.stat().st_size / 1e6:.2f} MB')

## Step 3 — Add lower-zoom overviews

CLI equivalent: `gdaladdo -r nearest map.mbtiles 2 4 8 16`

Overview factors `[2, 4, 8, 16]` create downsampled copies at 1/2, 1/4, 1/8, 1/16 of base resolution → effectively zoom levels (max-1), (max-2), (max-3), (max-4).

In [ ]:
with rasterio.open(mbtiles_path, 'r+') as ds:
    ds.build_overviews([2, 4, 8, 16], Resampling.nearest)
    ds.update_tags(ns='rio_overview', resampling='nearest')

print('Overviews built')
print(f'New size: {mbtiles_path.stat().st_size / 1e6:.2f} MB')

## Verify — gdalinfo equivalent

Inspect the resulting MBTiles file: bounds, CRS, zoom levels stored, and a quick visual.

In [ ]:
with rasterio.open(mbtiles_path) as ds:
    print(f'Driver:    {ds.driver}')
    print(f'CRS:       {ds.crs}')
    print(f'Bounds:    {ds.bounds}')
    print(f'Size:      {ds.width} x {ds.height}')
    print(f'Bands:     {ds.count} ({ds.dtypes})')
    print(f'Overviews per band: {[ds.overviews(i) for i in ds.indexes]}')
    print(f'Tags:      {ds.tags()}')

In [ ]:
# Read the MBTiles back and plot — confirms the file is valid and georeferenced
from rasterio.plot import show

fig, ax = plt.subplots(figsize=(8, 8))
with rasterio.open(mbtiles_path) as ds:
    show(ds, ax=ax)
test_3857.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=2)
ax.set_title(f'MBTiles round-trip ({mbtiles_path.name})')
ax.set_axis_off()
plt.show()

## Interactive view — open the MBTiles in a folium map

Two ways to view the MBTiles interactively:

1. **`folium.ImageOverlay`** — read the MBTiles as an image via rasterio, overlay it on a slippy map. Simple, no extra dependencies, works for any single tile pyramid.
2. **Direct tile serving** — would need `localtileserver` or a TiTiler-style server (not installed in this venv).

We use option 1 below.

In [ ]:
import folium
from folium.raster_layers import ImageOverlay
from rasterio.warp import transform_bounds

# Read the MBTiles raster and reproject bounds to WGS84 (folium uses lat/lon)
with rasterio.open(mbtiles_path) as ds:
    arr = ds.read()  # shape: (bands, h, w)
    src_crs = ds.crs
    src_bounds = ds.bounds  # in EPSG:3857

# Reorder to (h, w, bands) for image display
img_hw = arr.transpose(1, 2, 0)
if img_hw.shape[2] == 1:
    img_hw = img_hw[:, :, 0]

# Transform raster bounds → WGS84
w_lon, s_lat, e_lon, n_lat = transform_bounds(src_crs, 'EPSG:4326', *src_bounds)
center_lat = (s_lat + n_lat) / 2
center_lon = (w_lon + e_lon) / 2

# Sub-cell polygon in WGS84 for overlay
test_4326 = test_row.to_crs(epsg=4326)

m = folium.Map(location=[center_lat, center_lon], zoom_start=17, tiles='OpenStreetMap')

# Reference: Google Hybrid as a comparison basemap
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
    attr='Google Hybrid', name='Google Hybrid (online)',
).add_to(m)

# The MBTiles content as an ImageOverlay
ImageOverlay(
    image=img_hw,
    bounds=[[s_lat, w_lon], [n_lat, e_lon]],
    opacity=1.0,
    name=f'MBTiles ({mbtiles_path.name})',
    interactive=False,
    cross_origin=False,
    zindex=1,
).add_to(m)

# Sub-cell polygon outline
folium.GeoJson(
    test_4326.__geo_interface__,
    style_function=lambda x: {'color': 'red', 'weight': 3, 'fill': False},
    name='Sub-cell',
).add_to(m)

folium.LayerControl().add_to(m)

# Save to disk so it can also be opened standalone
html_path = OUT_DIR / 'mbtiles_preview.html'
m.save(str(html_path))
print(f'Saved interactive preview: {html_path}')
m

# Production-style MBTiles via `mbtiles_export`

The cells above prove the raw GeoTIFF → MBTiles → overviews pipeline works. Below, we test the **actual production helpers** in [`src/mapping/mbtiles_export.py`](../src/mapping/mbtiles_export.py) — the same functions [`scripts/generate_all_maps.py`](../scripts/generate_all_maps.py) calls when run with `--mbtiles`.

These mirror `MapGenerator.generate_overview` / `generate_detail` but burn the overlays directly into the raster pixels:

- **Overview (5km):** red cell boundary, primary sub-cells filled green, reserve filled yellow, red crosshair at each centroid
- **Detail (500m):** yellow building footprints, sub-cell boundary in green/goldenrod, red crosshair at the centroid (no "START" text — fixed-pixel labels don't survive zoom on a slippy map)

Roads, place names, hamlets, schools etc. come from Google Hybrid tiles for free.

In [ ]:
from src.mapping.mbtiles_export import export_detail_mbtiles, export_overview_mbtiles
from src.data_processing.load_boundaries import load_buildings, load_control_grid

buildings = load_buildings(data_dir)
if buildings is not None:
    buildings = validate_crs(buildings)
    print(f'Buildings loaded: {len(buildings):,}')

# Pick the same sub-cell we used above and its parent 5km cell
test_5km_id = test_row.iloc[0]['5km_id']
control_grid = load_control_grid(data_dir)
control_grid = validate_crs(control_grid)
test_cell = control_grid[control_grid['id'] == test_5km_id].copy()
test_cell_subcells = selected[selected['5km_id'] == test_5km_id].copy()
print(f'\nParent 5km cell: {test_5km_id}')
print(f'Selected sub-cells in cell: {len(test_cell_subcells)}')

In [ ]:
# Detail (500m sub-cell) MBTiles via the production function
role = test_row.iloc[0]['selection_role']
subcell_id = test_row.iloc[0].get('grid_id', '?')
building_count = test_row.iloc[0].get('building_count', '?')
detail_out = OUT_DIR / f'detail_{test_5km_id}_{role}.mbtiles'

detail_title = f'5km: {int(test_5km_id)} — 500m: {subcell_id} ({role}) — {building_count} buildings'

export_detail_mbtiles(
    subcell=test_row,
    output_path=detail_out,
    buildings=buildings,
    role=role,
    zoom=19,
    title=detail_title,
)

print(f'Wrote: {detail_out}')
print(f'Size:  {detail_out.stat().st_size / 1e6:.2f} MB')

with rasterio.open(detail_out) as ds:
    print(f'Bounds: {ds.bounds}')
    print(f'Size:   {ds.width} x {ds.height}')
    print(f'Overviews: {ds.overviews(1)}')

fig, ax = plt.subplots(figsize=(9, 9))
with rasterio.open(detail_out) as ds:
    show(ds, ax=ax)
ax.set_title(f'Detail MBTiles — sub-cell {test_5km_id} ({role})')
ax.set_axis_off()
plt.show()

In [ ]:
# Overview (5km cell) MBTiles via the production function
overview_out = OUT_DIR / f'overview_{test_5km_id}.mbtiles'
overview_ward = test_cell_subcells.iloc[0].get('ward_name', 'unknown')
overview_status = test_cell_subcells.iloc[0].get('sample_status', '?')
overview_title = f'5km cell {int(test_5km_id)} — {overview_ward} ({overview_status})'

export_overview_mbtiles(
    grid_cell=test_cell,
    selected_subcells=test_cell_subcells,
    output_path=overview_out,
    zoom=15,
    title=overview_title,
)

print(f'Wrote: {overview_out}')
print(f'Size:  {overview_out.stat().st_size / 1e6:.2f} MB')

with rasterio.open(overview_out) as ds:
    print(f'Bounds: {ds.bounds}')
    print(f'Size:   {ds.width} x {ds.height}')
    print(f'Overviews: {ds.overviews(1)}')

fig, ax = plt.subplots(figsize=(10, 10))
with rasterio.open(overview_out) as ds:
    show(ds, ax=ax)
ax.set_title(f'Overview MBTiles — 5km cell {test_5km_id}')
ax.set_axis_off()
plt.show()

## Next steps

If the styled detail + overview MBTiles look good above, run the production script with the `--mbtiles` flag — it will emit `overview_5km.mbtiles` and `subcell_<id>_<role>.mbtiles` next to the existing PNGs in every cell folder:

```bash
# Test on one cell first
python scripts/generate_all_maps.py --mbtiles --single 13151

# Or run for everything
python scripts/generate_all_maps.py --mbtiles

# Use --mbtiles-only to skip PNG generation if you want
python scripts/generate_all_maps.py --mbtiles-only

# Tune zoom (defaults: detail=19, overview=17). Lower = smaller files
python scripts/generate_all_maps.py --mbtiles --mbtiles-detail-zoom 18 --mbtiles-overview-zoom 16
```

Things still to confirm:

- **Test on a real SurveyCTO form** to confirm it loads as an offline basemap
- **File size at scale** — back-of-envelope: detail z=19 ≈ 1-2 MB each, overview z=17 ≈ 5-15 MB each. Multiply by ~600 cells × ~4 sub-cells = lots; consider whether `--mbtiles-detail-zoom 18` is good enough
- **Decide whether `generate_info_sheets.py` should also reference the `.mbtiles`** (e.g. as a download link) — currently it only embeds the PNGs as `<img>`